In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Pertemuan11-MLlibRegresi") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/12 16:54:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


In [2]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 2000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo"]

data = pd.DataFrame({
    "unit_terjual": np.random.randint(1, 15, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "rating": np.random.randint(1, 6, size=n),
})
# Total pendapatan dipengaruhi unit & harga (dengan sedikit variasi acak, seperti data real world)
data["pendapatan"] = data["unit_terjual"] * data["harga_satuan"] + np.random.normal(0, 20000, size=n)

df = spark.createDataFrame(data)
df.show(5)
print("Total baris:", df.count())

+------------+------------+------------+----------+------+------------------+
|unit_terjual|harga_satuan|    kategori|      kota|rating|        pendapatan|
+------------+------------+------------+----------+------+------------------+
|           7|      150000|     Fashion|  Magelang|     4| 997913.0612957058|
|           4|      100000|Rumah Tangga|  Semarang|     4| 389490.8317158621|
|          13|      150000|  Elektronik|Yogyakarta|     4|1936047.6709878189|
|          11|      100000|Rumah Tangga|  Magelang|     5|1080918.9109179082|
|           8|      150000|     Makanan|  Semarang|     2|1191696.4943767258|
+------------+------------+------------+----------+------+------------------+
only showing top 5 rows



[Stage 1:>                                                          (0 + 4) / 4]

Total baris: 2000


In [3]:
# Persiapan ulang untuk latihan
spark = SparkSession.builder.appName("Latihan11").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df = spark.createDataFrame(data)
print("Siap untuk latihan. Total baris:", df.count())

[Stage 4:==============>                                            (1 + 3) / 4]

Siap untuk latihan. Total baris: 2000


In [4]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

indexer_kategori = StringIndexer(inputCol="kategori", outputCol="kategori_idx")
indexer_kota = StringIndexer(inputCol="kota", outputCol="kota_idx")

df_indexed = indexer_kategori.fit(df).transform(df)
df_indexed = indexer_kota.fit(df_indexed).transform(df_indexed)

assembler_sederhana = VectorAssembler(
    inputCols=["unit_terjual", "harga_satuan", "rating"],
    outputCol="features_sederhana"
)
df_sederhana = assembler_sederhana.transform(df_indexed)

df_sederhana.select("unit_terjual", "harga_satuan", "rating", "features_sederhana").show(5, truncate=False)

+------------+------------+------+-------------------+
|unit_terjual|harga_satuan|rating|features_sederhana |
+------------+------------+------+-------------------+
|7           |150000      |4     |[7.0,150000.0,4.0] |
|4           |100000      |4     |[4.0,100000.0,4.0] |
|13          |150000      |4     |[13.0,150000.0,4.0]|
|11          |100000      |5     |[11.0,100000.0,5.0]|
|8           |150000      |2     |[8.0,150000.0,2.0] |
+------------+------------+------+-------------------+
only showing top 5 rows



In [5]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

assembler_utama = VectorAssembler(
    inputCols=["unit_terjual", "harga_satuan", "kategori_idx", "kota_idx", "rating"],
    outputCol="features"
)
df_final = assembler_utama.transform(df_sederhana)

train, test = df_final.randomSplit([0.8, 0.2], seed=42)

model_utama = LinearRegression(featuresCol="features", labelCol="pendapatan").fit(train)
model_sederhana = LinearRegression(featuresCol="features_sederhana", labelCol="pendapatan").fit(train)

pred_utama = model_utama.transform(test)
pred_sederhana = model_sederhana.transform(test)

ev_r2 = RegressionEvaluator(labelCol="pendapatan", predictionCol="prediction", metricName="r2")
r2_utama = ev_r2.evaluate(pred_utama)
r2_sed = ev_r2.evaluate(pred_sederhana)

print(f"R² model utama (5 fitur)     : {r2_utama:.4f}")
print(f"R² model sederhana (3 fitur) : {r2_sed:.4f}")
print(f"Selisih (sederhana - utama)  : {r2_sed - r2_utama:+.4f}")

[Stage 19:>                                                         (0 + 4) / 4]

R² model utama (5 fitur)     : 0.8781
R² model sederhana (3 fitur) : 0.8783
Selisih (sederhana - utama)  : +0.0002


In [7]:
from pyspark.sql.functions import col, abs as spark_abs, round as spark_round

pred_err = (
    pred_utama
    .withColumn("selisih", col("pendapatan") - col("prediction"))
    .withColumn("selisih_abs", spark_abs(col("selisih")))
)

pred_err.orderBy(col("selisih_abs").desc()).limit(5).select(
    "unit_terjual", "harga_satuan", "kategori", "kota", "rating",
    spark_round("pendapatan", 0).alias("pendapatan"),
    spark_round("prediction", 0).alias("prediction"),
    spark_round("selisih", 0).alias("selisih"),
    spark_round("selisih_abs", 0).alias("selisih_abs")
).show(truncate=False)

[Stage 20:>                                                         (0 + 4) / 4]

+------------+------------+----------+----------+------+----------+----------+---------+-----------+
|unit_terjual|harga_satuan|kategori  |kota      |rating|pendapatan|prediction|selisih  |selisih_abs|
+------------+------------+----------+----------+------+----------+----------+---------+-----------+
|1           |150000      |Elektronik|Solo      |1     |137034.0  |623361.0  |-486327.0|486327.0   |
|14          |150000      |Kesehatan |Magelang  |3     |2128127.0 |1664499.0 |463628.0 |463628.0   |
|14          |150000      |Makanan   |Magelang  |1     |2120244.0 |1664787.0 |455458.0 |455458.0   |
|1           |150000      |Elektronik|Solo      |1     |171412.0  |623361.0  |-451950.0|451950.0   |
|1           |150000      |Fashion   |Yogyakarta|3     |155319.0  |606227.0  |-450908.0|450908.0   |
+------------+------------+----------+----------+------+----------+----------+---------+-----------+



Spark MLlib mewajibkan fitur digabung menjadi satu kolom Vector agar semua algoritma memakai representasi yang seragam. Algoritma tidak harus mengetahui jumlah, nama, atau tipe kolom tiap dataset, dan bisa langsung menjalankan operasi aljabar linear per partisi di seluruh executor. Format ini juga mendukung vektor sparse untuk data berdimensi tinggi dan membuat tahap persiapan fitur bisa dirangkai dengan algoritma apa pun lewat antarmuka fit()/transform() yang sama. Di pandas/scikit-learn data ada dalam satu memori, sedangkan di Spark data tersebar di banyak node, sehingga antarmuka yang seragam dan efisien lebih penting.

In [8]:
spark.stop()